# Portuguese Speech-to-Text Model Testing
### Testing Local Models for European Portuguese (pt-PT)

This notebook tests multiple Portuguese-optimized STT models:
1. **Wav2Vec2 Portuguese models** (specifically trained on pt-PT)
2. **SeamlessM4T v2** (Meta's multilingual model)
3. **Faster-Whisper Large V3** (optimized inference)
4. **Portuguese-specific Whisper fine-tunes**

Upload your audio file (`Gravação2.m4a` or any test audio) and run all cells to compare results.

## Setup & Installation

In [1]:
# Install required packages
!pip install -q transformers torch torchaudio librosa soundfile accelerate
!pip install -q faster-whisper
!pip install -q git+https://github.com/huggingface/transformers.git  # Latest version for SeamlessM4T
!pip install -q pydub  # For audio conversion
!pip install kenlm
!pip install pyctcdecode

print("✅ All packages installed!")

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
✅ All packages installed!


In [2]:
import torch
import torchaudio
import librosa
import soundfile as sf
import numpy as np
from transformers import pipeline, AutoProcessor, AutoModelForSpeechSeq2Seq
from faster_whisper import WhisperModel
import time
from IPython.display import Audio, display
import warnings
warnings.filterwarnings('ignore')

# Check GPU availability
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"🚀 Using device: {device}")
if device == "cuda":
    print(f"   GPU: {torch.cuda.get_device_name(0)}")
    print(f"   Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

🚀 Using device: cuda
   GPU: Tesla T4
   Memory: 15.64 GB


In [22]:
from google.colab import drive
drive.mount('/content/drive',force_remount=True)

Mounted at /content/drive


In [31]:
!ls drive/MyDrive

Gravação2.m4a  Gravação.m4a


## Audio Preprocessing

Convert audio to the required format for each model

In [32]:
audio_path = "/content/drive/MyDrive/Gravação.m4a"

# Load and preprocess audio
def load_audio(file_path, target_sr=16000):
    """Load audio file and resample to target sample rate"""
    audio, sr = librosa.load(file_path, sr=target_sr, mono=True)
    return audio, sr

audio_array, sample_rate = load_audio(audio_path)
duration = len(audio_array) / sample_rate

print(f"Audio duration: {duration:.2f} seconds")
print(f"Sample rate: {sample_rate} Hz")
print(f"Audio shape: {audio_array.shape}")

Audio duration: 3.32 seconds
Sample rate: 16000 Hz
Audio shape: (53128,)


---
# Model Testing
---

## 1. Wav2Vec2 Portuguese Models

These models are specifically fine-tuned on Portuguese (including European Portuguese)

In [33]:
print("🔄 Loading Wav2Vec2 Portuguese models...\n")

wav2vec_models = [
    "jonatasgrosman/wav2vec2-large-xlsr-53-portuguese",  # Most popular
]

wav2vec_results = {}

for model_name in wav2vec_models:
    print(f"📊 Testing: {model_name}")
    try:
        start_time = time.time()
        
        # Load model
        pipe = pipeline(
            "automatic-speech-recognition",
            model=model_name,
            device=0 if device == "cuda" else -1,
            language="portuguese",
            prompt= f"Transcrição de uma chamada."
            f"O paciente fala em português de Portugal. "
            f"Termos frequentes: consulta, marcação, dentista, limpeza, obturação, "
            f"ortodontia, branqueamento, coroa, seguro, ADSE, Multicare, Médis, "
            f"cancelar, remarcar, urgência, dor de dentes, receção, check-up. "
        )

        # Transcribe
        result = pipe(audio_array, chunk_length_s=30, stride_length_s=5)
        
        elapsed = time.time() - start_time
        
        wav2vec_results[model_name] = {
            "text": result["text"],
            "time": elapsed
        }
        
        print(f"   ⏱️  Time: {elapsed:.2f}s")
        print(f"   📝 Transcription: {result['text']}")
        print()
        
        # Free memory
        del pipe
        torch.cuda.empty_cache() if device == "cuda" else None
        
    except Exception as e:
        print(f"   ❌ Error: {str(e)}")
        print()

print("✅ Wav2Vec2 testing complete!")

🔄 Loading Wav2Vec2 Portuguese models...

📊 Testing: jonatasgrosman/wav2vec2-large-xlsr-53-portuguese


Loading weights:   0%|          | 0/424 [00:00<?, ?it/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

   ⏱️  Time: 33.06s
   📝 Transcription: oh está madura no dente ser e se uma cama consulta

✅ Wav2Vec2 testing complete!


## 2. SeamlessM4T v2 (Meta)

Meta's state-of-the-art multilingual model with strong Portuguese support

In [34]:
print("🔄 Loading SeamlessM4T v2...\n")

try:
    from transformers import SeamlessM4Tv2ForSpeechToText, AutoProcessor
    
    model_name = "facebook/seamless-m4t-v2-large"
    
    start_time = time.time()
    
    processor = AutoProcessor.from_pretrained(model_name)
    model = SeamlessM4Tv2ForSpeechToText.from_pretrained(model_name)
    
    if device == "cuda":
        model = model.to("cuda")
    
    # Process audio
    inputs = processor(audio=audio_array, sampling_rate=sample_rate, return_tensors="pt")
    
    if device == "cuda":
        inputs = {k: v.to("cuda") for k, v in inputs.items()}
    
    # Generate transcription (Portuguese)
    output_tokens = model.generate(**inputs, tgt_lang="por", max_new_tokens=256)
    transcription = processor.decode(output_tokens[0].tolist(), skip_special_tokens=True)
    
    elapsed = time.time() - start_time
    
    print(f"📊 SeamlessM4T v2 Results:")
    print(f"   ⏱️  Time: {elapsed:.2f}s")
    print(f"   📝 Transcription: {transcription}")
    
    seamless_result = {
        "text": transcription,
        "time": elapsed
    }
    
    # Free memory
    del model, processor
    torch.cuda.empty_cache() if device == "cuda" else None
    
    print("\n✅ SeamlessM4T testing complete!")
    
except Exception as e:
    print(f"❌ Error with SeamlessM4T: {str(e)}")
    print("This might require the latest transformers version.")
    seamless_result = None

🔄 Loading SeamlessM4T v2...



Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/1429 [00:00<?, ?it/s]

SeamlessM4Tv2ForSpeechToText LOAD REPORT from: facebook/seamless-m4t-v2-large
Key                                                                           | Status     |  | 
------------------------------------------------------------------------------+------------+--+-
vocoder.hifi_gan.resblocks.{0...14}.convs1.{0, 1, 2}.weight                   | UNEXPECTED |  | 
t2u_model.model.encoder.layers.{0, 1, 2, 3, 4, 5}.self_attn.k_proj.bias       | UNEXPECTED |  | 
vocoder.hifi_gan.resblocks.{0...14}.convs1.{0, 1, 2}.bias                     | UNEXPECTED |  | 
t2u_model.model.encoder.layers.{0, 1, 2, 3, 4, 5}.self_attn.v_proj.weight     | UNEXPECTED |  | 
t2u_model.model.decoder.layers.{0, 1, 2, 3, 4, 5}.self_attn_layer_norm.bias   | UNEXPECTED |  | 
vocoder.hifi_gan.resblocks.{0...14}.convs2.{0, 1, 2}.bias                     | UNEXPECTED |  | 
text_encoder.layers.{0...23}.self_attn.q_proj.bias                            | UNEXPECTED |  | 
text_encoder.layers.{0...23}.self_attn.q_proj.wei

📊 SeamlessM4T v2 Results:
   ⏱️  Time: 38.79s
   📝 Transcription: Olá, estamos dando um dente, sempre se eu marcar uma consulta.

✅ SeamlessM4T testing complete!


## 3. Faster-Whisper Large V3

Optimized Whisper implementation (4x faster, less memory)

In [36]:
print("🔄 Loading Faster-Whisper Large V3...\n")

# Dental clinic prompt (same as your current setup)
dental_prompt = (
    "Transcrição de uma chamada para a Clínica Dentária Sol Nascente. "
    "O paciente fala em português de Portugal."
    "Termos frequentes: consulta, marcação, dentista, limpeza, obturação, "
    "ortodontia, branqueamento, coroa, seguro, ADSE, Multicare, Médis, "
    "cancelar, remarcar, urgência, dor de dentes, receção, check-up."
)

whisper_models = [
    "large-v3",
]

faster_whisper_results = {}

for model_size in whisper_models:
    print(f"📊 Testing: Faster-Whisper {model_size}")
    try:
        start_time = time.time()
        
        # Load model with optimizations
        model = WhisperModel(
            model_size,
            device="cuda" if device == "cuda" else "cpu",
            compute_type="float16" if device == "cuda" else "int8"
        )
        
        # Transcribe with prompt
        segments, info = model.transcribe(
            audio_array,
            language="pt",
            initial_prompt=dental_prompt,
            beam_size=10,
            best_of=5,
            temperature=0.0,
            vad_filter=True,  # Voice activity detection
            vad_parameters=dict(min_silence_duration_ms=500)
        )
        
        # Combine segments
        transcription = " ".join([segment.text for segment in segments])
        
        elapsed = time.time() - start_time
        
        faster_whisper_results[model_size] = {
            "text": transcription.strip(),
            "time": elapsed,
            "language_probability": info.language_probability
        }
        
        print(f"   ⏱️  Time: {elapsed:.2f}s")
        print(f"   🌍 Language confidence: {info.language_probability:.2%}")
        print(f"   📝 Transcription: {transcription.strip()}")
        print()
        
        # Free memory
        del model
        torch.cuda.empty_cache() if device == "cuda" else None
        
    except Exception as e:
        print(f"   ❌ Error: {str(e)}")
        print()

print("✅ Faster-Whisper testing complete!")

🔄 Loading Faster-Whisper Large V3...

📊 Testing: Faster-Whisper large-v3
   ⏱️  Time: 10.76s
   🌍 Language confidence: 100.00%
   📝 Transcription: Ó, está-me a dor no dente, sempre se eu marcar uma consulta.

✅ Faster-Whisper testing complete!


In [9]:
[segment.text for segment in segments]

[]

## 4. Portuguese Fine-tuned Whisper Models

Community fine-tuned Whisper models specifically for Portuguese

In [5]:
print("🔄 Loading Portuguese Fine-tuned Whisper models...\n")

pt_whisper_models = [
    "pierreguillou/whisper-medium-portuguese"
]

pt_whisper_results = {}

for model_name in pt_whisper_models:
    print(f"📊 Testing: {model_name}")
    try:
        start_time = time.time()
        
        pipe = pipeline(
            "automatic-speech-recognition",
            model=model_name,
            device=0 if device == "cuda" else -1,
            torch_dtype=torch.float16 if device == "cuda" else torch.float32
        )
        
        result = pipe(
            audio_array,
            generate_kwargs={
                "language": "portuguese",
                "task": "transcribe"
            },
            chunk_length_s=30,
            stride_length_s=5
        )
        
        elapsed = time.time() - start_time
        
        pt_whisper_results[model_name] = {
            "text": result["text"],
            "time": elapsed
        }
        
        print(f"   ⏱️  Time: {elapsed:.2f}s")
        print(f"   📝 Transcription: {result['text']}")
        print()
        
        # Free memory
        del pipe
        torch.cuda.empty_cache() if device == "cuda" else None
        
    except Exception as e:
        print(f"   ❌ Error: {str(e)}")
        print()

print("✅ Portuguese Whisper testing complete!")

🔄 Loading Portuguese Fine-tuned Whisper models...

📊 Testing: pierreguillou/whisper-medium-portuguese


`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/3.06G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/947 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/830 [00:00<?, ?B/s]

...

added_tokens.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

preprocessor_config.json: 0.00B [00:00, ?B/s]

Using `chunk_length_s` is very experimental with seq2seq models. The results will not necessarily be entirely accurate and will have caveats. More information: https://github.com/huggingface/transformers/pull/20104. Ignore this warning with pipeline(..., ignore_warning=True). To use Whisper for long-form transcription, use rather the model's `generate` method directly as the model relies on it's own chunking mechanism (cf. Whisper original paper, section 3.8. Long-form Transcription).


   ❌ Error: The generation config is outdated and is thus not compatible with the `language` argument to `generate`. Please update the generation config as per the instructions https://github.com/huggingface/transformers/issues/25084#issuecomment-1664398224

✅ Portuguese Whisper testing complete!


---
# Results Comparison
---

In [36]:
print("="*80)
print("📊 FINAL COMPARISON")
print("="*80)
print()

all_results = []

# Wav2Vec2 results
for model_name, result in wav2vec_results.items():
    all_results.append({
        "Model Type": "Wav2Vec2",
        "Model": model_name.split("/")[-1],
        "Time (s)": f"{result['time']:.2f}",
        "Transcription": result['text'][:100] + "..." if len(result['text']) > 100 else result['text']
    })

# SeamlessM4T result
if seamless_result:
    all_results.append({
        "Model Type": "SeamlessM4T",
        "Model": "v2-large",
        "Time (s)": f"{seamless_result['time']:.2f}",
        "Transcription": seamless_result['text'][:100] + "..." if len(seamless_result['text']) > 100 else seamless_result['text']
    })

# Faster-Whisper results
for model_size, result in faster_whisper_results.items():
    all_results.append({
        "Model Type": "Faster-Whisper",
        "Model": model_size,
        "Time (s)": f"{result['time']:.2f}",
        "Transcription": result['text'][:100] + "..." if len(result['text']) > 100 else result['text']
    })

# PT Whisper results
for model_name, result in pt_whisper_results.items():
    all_results.append({
        "Model Type": "PT-Whisper",
        "Model": "whisper-large-v3-pt",
        "Time (s)": f"{result['time']:.2f}",
        "Transcription": result['text'][:100] + "..." if len(result['text']) > 100 else result['text']
    })

# Display results
import pandas as pd
df = pd.DataFrame(all_results)
print(df.to_string(index=False))
print()
print("="*80)

# Show full transcriptions
print("\n📝 FULL TRANSCRIPTIONS:\n")
print("="*80)

print("\n🔹 WAV2VEC2 MODELS:")
for model_name, result in wav2vec_results.items():
    print(f"\n{model_name}:")
    print(f"{result['text']}")
    print("-"*80)

if seamless_result:
    print("\n🔹 SEAMLESSM4T V2:")
    print(f"{seamless_result['text']}")
    print("-"*80)

print("\n🔹 FASTER-WHISPER:")
for model_size, result in faster_whisper_results.items():
    print(f"\n{model_size}:")
    print(f"{result['text']}")
    print("-"*80)

print("\n🔹 PORTUGUESE FINE-TUNED WHISPER:")
for model_name, result in pt_whisper_results.items():
    print(f"\n{model_name}:")
    print(f"{result['text']}")
    print("-"*80)

📊 FINAL COMPARISON

    Model Type                             Model Time (s)                                 Transcription
      Wav2Vec2 wav2vec2-large-xlsr-53-portuguese    33.71 olá vai modelo que posso marcar uma com solta
Faster-Whisper                          large-v3    59.86      Olá, Diamond. Posso marcar uma consulta?
Faster-Whisper                          large-v2   103.29            Obrigado e até a próxima consulta.


📝 FULL TRANSCRIPTIONS:


🔹 WAV2VEC2 MODELS:

jonatasgrosman/wav2vec2-large-xlsr-53-portuguese:
olá vai modelo que posso marcar uma com solta
--------------------------------------------------------------------------------

🔹 FASTER-WHISPER:

large-v3:
Olá, Diamond. Posso marcar uma consulta?
--------------------------------------------------------------------------------

large-v2:
Obrigado e até a próxima consulta.
--------------------------------------------------------------------------------

🔹 PORTUGUESE FINE-TUNED WHISPER:


## Save Results

In [ ]:
import json

# Compile all results
final_results = {
    "audio_file": audio_path,
    "audio_duration": duration,
    "wav2vec2": wav2vec_results,
    "seamless_m4t": seamless_result,
    "faster_whisper": faster_whisper_results,
    "portuguese_whisper": pt_whisper_results
}

# Save to JSON
with open('transcription_results.json', 'w', encoding='utf-8') as f:
    json.dump(final_results, f, ensure_ascii=False, indent=2)

print("✅ Results saved to 'transcription_results.json'")

# Download results
files.download('transcription_results.json')
print("📥 Results file downloaded!")

---
# Bonus: Test with Multiple Audio Files
---

Upload multiple audio files to compare model consistency

In [ ]:
# Optional: Test with multiple files
print("📁 Upload multiple audio files for batch testing:")
uploaded_batch = files.upload()

batch_results = {}

for filename in uploaded_batch.keys():
    print(f"\n{'='*80}")
    print(f"Testing: {filename}")
    print(f"{'='*80}\n")
    
    audio_array, sr = load_audio(filename)
    
    # Test with your preferred model(s)
    # Add your testing code here based on results above
    
print("✅ Batch testing complete!")